# 04 — SHAP Prototyping on CMAPSS

**Goal:** Train a classifier on CMAPSS FD001 and apply KernelSHAP to explain which sensors at which timesteps drive failure predictions.

**Model:** sklearn MLP (proxy for LSTM — same concept, runs without PyTorch)  
**SHAP method:** KernelSHAP (baseline — TimeSHAP and WinIT next)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from src.data.cmapss_loader import CMAPSSLoader, CONSTANT_SENSORS, CMAPSS_COLUMNS
from src.data.preprocessor import TimeSeriesPreprocessor

print('Imports OK')


## 1. Load and Prepare Data

In [ ]:
loader = CMAPSSLoader(data_dir='../data/raw/cmapss/CMaps')
train_df, test_df = loader.load(subset='FD001')

train_df = loader.add_failure_label(train_df, rul_threshold=30)
test_df  = loader.add_failure_label(test_df,  rul_threshold=30)
train_df = loader.drop_constant_sensors(train_df)
test_df  = loader.drop_constant_sensors(test_df)

sensor_cols    = [c for c in CMAPSS_COLUMNS if c.startswith('sensor')]
active_sensors = [c for c in sensor_cols if c not in CONSTANT_SENSORS]

prep = TimeSeriesPreprocessor()
train_norm = prep.normalise(train_df, method='minmax', feature_cols=active_sensors)
test_norm  = prep.transform(test_df,  feature_cols=active_sensors)

WINDOW = 30
X_list, y_list = [], []
for uid in train_norm['unit_id'].unique():
    unit_df = train_norm[train_norm['unit_id'] == uid]
    X_u, y_u = prep.create_windows(unit_df, active_sensors, 'failure_imminent',
                                   window_size=WINDOW, step=1)
    X_list.append(X_u); y_list.append(y_u)

X_train_3d = np.concatenate(X_list)
y_train    = np.concatenate(y_list)

X_list_t, y_list_t = [], []
for uid in test_norm['unit_id'].unique():
    unit_df = test_norm[test_norm['unit_id'] == uid]
    X_u, y_u = prep.create_windows(unit_df, active_sensors, 'failure_imminent',
                                   window_size=WINDOW, step=1)
    X_list_t.append(X_u); y_list_t.append(y_u)

X_test_3d = np.concatenate(X_list_t)
y_test    = np.concatenate(y_list_t)

# Flatten (n, 30, 15) -> (n, 450) for MLP
X_train = X_train_3d.reshape(len(X_train_3d), -1)
X_test  = X_test_3d.reshape(len(X_test_3d),  -1)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Class balance (train) — Normal: {(y_train==0).sum():,}  |  Failure: {(y_train==1).sum():,}')


## 2. Train MLP Classifier

In [ ]:
model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    max_iter=50,
    random_state=42,
    verbose=True,
    early_stopping=True,
    validation_fraction=0.1,
)
model.fit(X_train, y_train)
print('Training complete')


In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Normal', 'Failure']))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=['Normal', 'Failure']).plot(ax=ax, colorbar=False)
ax.set_title('Confusion Matrix — CMAPSS FD001')
plt.tight_layout(); plt.show()


## 3. KernelSHAP — Feature Importance

In [ ]:
# Use 100 background samples and explain 50 test samples (KernelSHAP is slow)
rng = np.random.default_rng(42)
bg_idx   = rng.choice(len(X_train), size=100, replace=False)
exp_idx  = rng.choice(len(X_test),  size=50,  replace=False)

background = X_train[bg_idx]
X_explain  = X_test[exp_idx]

print('Running KernelSHAP (this takes ~2-3 minutes)...')
explainer   = shap.KernelExplainer(model.predict_proba, background)
shap_raw = explainer.shap_values(X_explain, nsamples=100, silent=True)

# Handle both list [class0, class1] and single array depending on shap version
if isinstance(shap_raw, list):
    sv_failure = np.array(shap_raw[1])
else:
    sv_failure = np.array(shap_raw)
print(f'SHAP values shape: {sv_failure.shape}')


## 4. Global Feature Importance (per sensor)

In [ ]:
# Reshape back to (n_samples, timesteps, features) then average over samples and timesteps
sv_3d = sv_failure.reshape(sv_failure.shape[0], WINDOW, len(active_sensors))

# Mean absolute SHAP per sensor (averaged over timesteps and samples)
sensor_importance = np.abs(sv_3d).mean(axis=(0, 1))   # shape (15,)

order = np.argsort(sensor_importance)[::-1]
sorted_sensors = [active_sensors[i] for i in order]
sorted_values  = sensor_importance[order]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(sorted_sensors[::-1], sorted_values[::-1], color='steelblue')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('KernelSHAP — Global Sensor Importance (CMAPSS FD001)')
plt.tight_layout(); plt.show()

print('Top 5 most important sensors:')
for i in range(5):
    print(f'  {i+1}. {sorted_sensors[i]}  —  {sorted_values[i]:.5f}')


## 5. Temporal Importance — Which Timesteps Matter?

In [ ]:
# Mean absolute SHAP per timestep (averaged over samples and features)
timestep_importance = np.abs(sv_3d).mean(axis=(0, 2))   # shape (30,)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(WINDOW), timestep_importance, marker='o', markersize=4, color='steelblue')
ax.fill_between(range(WINDOW), timestep_importance, alpha=0.3, color='steelblue')
ax.set_xlabel('Timestep within window (0 = oldest, 29 = most recent)')
ax.set_ylabel('Mean |SHAP value|')
ax.set_title('KernelSHAP — Temporal Importance (which timesteps matter most?)')
plt.tight_layout(); plt.show()


## 6. SHAP Heatmap — Sensor x Timestep

In [ ]:
# Average absolute SHAP over all explained samples -> (30, 15) matrix
heatmap_data = np.abs(sv_3d).mean(axis=0).T   # shape (15, 30)

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(heatmap_data, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_yticks(range(len(active_sensors)))
ax.set_yticklabels(active_sensors, fontsize=9)
ax.set_xlabel('Timestep within window (0 = oldest, 29 = most recent)')
ax.set_title('KernelSHAP Heatmap — Sensor x Timestep importance for failure prediction')
plt.colorbar(im, ax=ax, label='Mean |SHAP value|')
plt.tight_layout(); plt.show()


## 7. Single Prediction Explanation

In [ ]:
# Pick one failure prediction and explain it
failure_preds = np.where((model.predict(X_explain) == 1) & (y_test[exp_idx] == 1))[0]
if len(failure_preds) == 0:
    idx = 0
else:
    idx = failure_preds[0]

single_sv = sv_3d[idx]   # shape (30, 15)

# Top contributing sensor for this prediction
top_sensor_idx = np.abs(single_sv).mean(axis=0).argmax()
top_sensor     = active_sensors[top_sensor_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: SHAP over time for the top sensor
axes[0].plot(single_sv[:, top_sensor_idx], marker='o', markersize=4, color='tomato')
axes[0].axhline(0, color='grey', linestyle='--', linewidth=0.8)
axes[0].set_title(f'SHAP over time — {top_sensor} (most important sensor)')
axes[0].set_xlabel('Timestep'); axes[0].set_ylabel('SHAP value')

# Right: heatmap for this single prediction
im2 = axes[1].imshow(single_sv.T, aspect='auto', cmap='coolwarm',
                      norm=mcolors.CenteredNorm(), interpolation='nearest')
axes[1].set_yticks(range(len(active_sensors)))
axes[1].set_yticklabels(active_sensors, fontsize=8)
axes[1].set_xlabel('Timestep')
axes[1].set_title('SHAP heatmap — single failure prediction')
plt.colorbar(im2, ax=axes[1], label='SHAP value')

plt.tight_layout(); plt.show()
print(f'Prediction: {model.predict(X_explain[[idx]])[0]}  |  True label: {y_test[exp_idx[idx]]}')
print(f'Most important sensor: {top_sensor}')


## Summary

| Step | Result |
|------|--------|
| Model | MLP (128→64), trained on CMAPSS FD001 |
| SHAP method | KernelSHAP baseline |
| What we learned | Which sensors and which timesteps drive failure predictions |
| Expected top sensors | T30, P30, phi (HPC degradation signatures) |

**Next steps:**
- Replace MLP with LSTM (PyTorch) once environment is set up
- Implement TimeSHAP — compare temporal importance curves
- Implement WinIT — compare against KernelSHAP and TimeSHAP
- Apply same pipeline to real robot data